# Recipe data cleaning

Load, inspect, and clean `recipes.csv` using **pandas**, **numpy**, and **matplotlib** only.

Target columns: `RecipeId`, `Name`, `RecipeCategory`, `RecipeIngredientParts`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

TARGET_COLUMNS = [
    "RecipeId",
    "Name",
    "RecipeCategory",
    "RecipeIngredientParts",
]
TEXT_COLUMNS = ["Name", "RecipeCategory", "RecipeIngredientParts"]

## 1. Data ingestion

In [ ]:
recipes = pd.read_csv("recipes.csv", usecols=TARGET_COLUMNS)

print("Initial shape:", recipes.shape)
print()
recipes.info()

## 2. Missing values

In [ ]:
print(recipes.isna().sum())

In [ ]:
missing_ids = int(recipes["RecipeId"].isna().sum())
if missing_ids:
    recipes = recipes.dropna(subset=["RecipeId"])
    print(f"Dropped {missing_ids} row(s) with missing RecipeId.")
else:
    print("No missing RecipeId values.")

recipes["Name"] = recipes["Name"].fillna("")
recipes["RecipeIngredientParts"] = recipes["RecipeIngredientParts"].fillna("")
recipes["RecipeCategory"] = recipes["RecipeCategory"].fillna("Unknown")

## 3. String cleaning and normalization

In [ ]:
for col in TEXT_COLUMNS:
    recipes[col] = (
        recipes[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

recipes["RecipeCategory"] = recipes["RecipeCategory"].str.title()

## 4. Structural and data type integrity

In [ ]:
recipes["RecipeId"] = pd.to_numeric(recipes["RecipeId"], errors="coerce")
invalid_ids = int(recipes["RecipeId"].isna().sum())
if invalid_ids:
    recipes = recipes.dropna(subset=["RecipeId"])
    print(f"Dropped {invalid_ids} row(s) with invalid RecipeId.")

recipes["RecipeId"] = recipes["RecipeId"].astype(np.int64)

duplicate_count = int(recipes["RecipeId"].duplicated().sum())
recipes = recipes.drop_duplicates(subset=["RecipeId"], keep="first").reset_index(drop=True)

if duplicate_count:
    print(f"Removed {duplicate_count} duplicate RecipeId row(s).")
else:
    print("No duplicate RecipeId values.")

## 5. Inspection and sanity check

In [ ]:
print("Final shape:", recipes.shape)
print()
print(recipes.isna().sum())
print()
recipes.head(10)

In [ ]:
top10 = recipes["RecipeCategory"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(top10.index.astype(str), top10.to_numpy())
ax.set_title("Top 10 Recipe Categories")
ax.set_xlabel("RecipeCategory")
ax.set_ylabel("Number of recipes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()